In [1]:
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

# --- RICH & METRIC IMPORTS ---
from rich.console import Console
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeRemainingColumn
from rich.table import Table
from torchmetrics import MetricCollection
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

# --- YOUR CUSTOM IMPORTS (Assumed available) ---
from model import FM_PhysMamba_UNET, ODESolver
from data.utils import get_haze_transforms, restandardize_tensor
from data import RESIDE_SOTS_Indoor
from utils import pad_to_multiple, unpad


In [2]:
# --- CONFIGURATION ---
WEIGHT_PATH = "checkpoints/FM_PhysicMamba_UNET_DDP/stage_0_finished.pt"
DATASET_ROOT = "dataset"
OUTPUT_DIR = "evaluation_results"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 256
NFE = 10  # Number of steps for the solver

### Loading Dataset for evaluation

In [3]:
# --- VISUALIZATION FUNCTION ---
def save_comparison(hazy, pred, clean, idx, save_dir = OUTPUT_DIR):
    """
    Saves a grid: Hazy | Prediction | Ground Truth | Error Map
    """
    # Convert to numpy (H, W, C)
    def to_np(t): return t.squeeze().permute(1, 2, 0).cpu().numpy().clip(0, 1)
    
    h_img = to_np(hazy)
    p_img = to_np(pred)
    c_img = to_np(clean)
    
    # Calculate Error Map (Heatmap of differences)
    diff = np.abs(c_img - p_img)
    error_map = np.mean(diff, axis=2) # Grayscale error
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    titles = ["Input (Hazy)", "Flow Matching (Ours)", "Ground Truth", "Error Map"]
    images = [h_img, p_img, c_img, error_map]
    
    for ax, img, title in zip(axes, images, titles):
        if title == "Error Map":
            im = ax.imshow(img, cmap='jet', vmin=0, vmax=0.2) # Jet colormap for error
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        else:
            ax.imshow(img)
        ax.set_title(title, fontsize=14)
        ax.axis('off')
    
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"eval_{idx:04d}.png")
    plt.savefig(save_path)

    plt.close()

In [6]:
@torch.no_grad()
def evaluation(model, solver, loader, device="cuda", nfe=10, console=None, step=0):
    model.eval()

    # Metrics
    eval_metrics = MetricCollection({
        "PSNR": PeakSignalNoiseRatio(data_range=1.0),
        "SSIM": StructuralSimilarityIndexMeasure(data_range=1.0),
        "LPIPS": LearnedPerceptualImagePatchSimilarity(net_type='alex', normalize=True)
    }).to(device)

    eval_metrics.reset()

    # Progress Bar
    progress = Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(),
        TextColumn("[progress.percentage]{task.percentage:>3.0f}%"),
        TimeRemainingColumn(),
        console=console
    )

    visuals_buffer = {"hazy": [], "clean": [], "pred": []}
    capture_limit = 4
    captured_count = 0

    progress.start()
    task_id = progress.add_task("Evaluating...", total=len(loader))

    for batch_idx, batch in enumerate(loader):
        clean_img, hazy_img = batch
        clean_img = clean_img.to(device, non_blocking=True)
        hazy_img = hazy_img.to(device, non_blocking=True)
        
        # Pad
        hazy_padded, pad_h, pad_w = pad_to_multiple(hazy_img, multiple=16)
        
        # Inference
        with torch.amp.autocast("cuda"):
            # FIX: Use 'solver', not 'self.ode_solver'
            pred_padded = solver.sample(hazy_padded, nfe=nfe) 
        
        pred_raw = unpad(pred_padded, pad_h, pad_w)

        # Post-process
        pred_final = restandardize_tensor(pred_raw)
        clean_final = restandardize_tensor(clean_img)
        hazy_final = restandardize_tensor(hazy_img)
        
        pred_clean = torch.clamp(pred_final, 0.0, 1.0)
        clean_target = torch.clamp(clean_final, 0.0, 1.0)
        
        eval_metrics.update(pred_clean, clean_target)

        # Capture Visuals
        if captured_count < capture_limit:
            needed = capture_limit - captured_count
            available = clean_final.shape[0]
            take = min(needed, available)
            
            visuals_buffer["hazy"].append(hazy_final[:take].cpu())
            visuals_buffer["clean"].append(clean_final[:take].cpu())
            visuals_buffer["pred"].append(pred_final[:take].cpu())
            
            captured_count += take
            
        progress.update(task_id, advance=1)

        if batch_idx == 10:
            break

    progress.stop()

    # Save Visuals
    if len(visuals_buffer["hazy"]) > 0:
        hazy_cat = torch.cat(visuals_buffer["hazy"], dim=0)
        clean_cat = torch.cat(visuals_buffer["clean"], dim=0)
        pred_cat = torch.cat(visuals_buffer["pred"], dim=0)
        
        save_comparison(hazy_cat, pred_cat, clean_cat, idx = step)    

    # Compute Results
    results = eval_metrics.compute()
    return results

### Load the model

In [7]:
# --- 1. SETUP CONSOLE ---
console = Console()

console.rule("[bold red]FM-PhysMamba Evaluation System[/bold red]")
    
# A. Load Model
console.print(f"[green]Loading weights from:[/green] {WEIGHT_PATH}")
checkpoint = torch.load(WEIGHT_PATH, map_location=DEVICE, weights_only=False)
state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint

# Clean 'module.' prefix if trained with DDP
new_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

model = FM_PhysMamba_UNET("small").to(DEVICE)
model.load_state_dict(new_state_dict, strict=True)
solver = ODESolver(model)


# Load Data
# evaluation_set = 
console.print(f"[green]Loading Dataset:[/green] RESIDE SOTS Indoor")
val_transform = get_haze_transforms(dataset_name="RESIDE", resize_size=RESOLUTION, split="val")
val_dataset = RESIDE_SOTS_Indoor(
    dataset_path=os.path.join(DATASET_ROOT, "reside-sots"),
    transform=val_transform,
    metadata="metadata_indoor.csv"
)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=4)

# D. Run Evaluation
metrics = evaluation(model, solver, val_loader, device=DEVICE, nfe=NFE, console=console)

# E. Print Final Table
table = Table(title="Final Performance Metrics")
table.add_column("Metric", style="cyan", no_wrap=True)
table.add_column("Value", style="magenta")

for k, v in metrics.items():
    table.add_row(k, f"{v.item():.4f}")
    
console.print(table)
console.print(f"\n[bold white]Visualizations saved to: {OUTPUT_DIR}[/bold white]")

───────────────────────────────────────── FM-PhysMamba Evaluation System ──────────────────────────────────────────

Loading weights from: checkpoints/FM_PhysicMamba_UNET_DDP/stage_0_finished.pt

Loading Dataset: RESIDE SOTS Indoor

Output()

RuntimeError: permute(sparse_coo): number of dimensions in the tensor input does not match the length of the desired ordering of dimensions i.e. input.dim() = 4 is not equal to len(dims) = 3

In [ ]:
console.print(f"[green]Loading Dataset:[/green] RESIDE SOTS Outdoors")
val_transform = get_haze_transforms(dataset_name="RESIDE", resize_size=RESOLUTION, split="val")
val_dataset = RESIDE_SOTS_Indoor(
    dataset_path=os.path.join(DATASET_ROOT, "reside-sots"),
    transform=val_transform,
    metadata="metadata_indoor.csv"
)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=4)

# D. Run Evaluation
metrics = evaluation(model, solver, val_loader, device=DEVICE, nfe=NFE, console=console)

# E. Print Final Table
table = Table(title="Final Performance Metrics")
table.add_column("Metric", style="cyan", no_wrap=True)
table.add_column("Value", style="magenta")

for k, v in metrics.items():
    table.add_row(k, f"{v.item():.4f}")
    
console.print(table)
console.print(f"\n[bold white]Visualizations saved to: {OUTPUT_DIR}[/bold white]")